# 3. The crowd model
**Two facts about the real data shape the method:** (1) about half of the tickets have no `vehicle_number`, so trips cannot be rebuilt; (2) the file holds only about one ticketed passenger per scheduled trip per day, so it is a **thin sample** of real ridership.

So we measure **flow**: ticketed passengers whose ride covers a stretch, divided by the number of scheduled trips (from GTFS) and the number of days. That is the average ticketed passengers on board per bus on that stretch.

In [ ]:
import sys, os
sys.path.append("..")                       # lets the notebook import config.py and the crowd/ folder
# os.environ["DATA_MODE"] = "synthetic"     # <- uncomment to try everything on the demo data
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import config
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
P = config.PROC; print("mode:", config.MODE, "| processed folder:", P)

In [ ]:
from crowd import load, flow, model
from crowd.text import level_of
# The idea on a tiny example: 3 passengers, 6 stops. Boarding adds, alighting subtracts, np.cumsum gives the load.
board = np.array([0, 1, 1]); alight = np.array([3, 4, 2]); pax = np.array([2, 1, 1])
L = load.load_along_trip(board, alight, pax, n_stops=6)
print('load between stops:', L)
plt.step(range(len(L)), L, where='post'); plt.xlabel('stretch (stop i -> i+1)'); plt.ylabel('passengers on board'); plt.show()

In [ ]:
tx = pd.read_parquet(P / 'tickets_resolved.parquet'); pat = pd.read_parquet(P / 'patterns.parquet'); hourly = pd.read_parquet(P / 'route_hourly.parquet')
d = tx.groupby('date').agg(pax=('pax', 'sum'), dow=('dow', 'first')).reset_index(); wk = d[d.dow < 6].pax; med = wk.median(); mad = (wk - med).abs().median()   # judge weekdays against weekdays
excl = d[(d.dow < 6) & ((d.pax - med).abs() > 3.5 * 1.4826 * mad)].date.tolist(); print('holiday-like weekdays left out:', excl)
cells_raw = flow.compute_flow_cells(tx, hourly, pat, exclude_dates=excl)
print(cells_raw.shape); cells_raw.head()

`tix_load` = ticketed passengers on board per scheduled trip. `n_tickets` = how many tickets cover that stretch (our evidence). Cells with few tickets are noisy, so we **shrink** them toward their route's average: `est = (n·x + k·prior) / (n + k)`.

In [ ]:
attrs = tx.groupby('route_n').agg(route=('route_raw', 'first'), is_ac=('is_ac', lambda s: int(s.mean() > 0.5))).reset_index(); attrs['route'] = attrs.route.astype(str)
cells, lv, cal = model.build_crowd_tables(cells_raw, attrs)
print(cal)
thin = cells[cells.n_tickets < 20].sample(min(5000, (cells.n_tickets < 20).sum()), random_state=0)
sns.scatterplot(data=thin, x='tix_load', y='est_load', alpha=.4, s=10); m = thin.tix_load.max(); plt.plot([0, m], [0, m], 'k--'); plt.title('Shrinkage on thin cells (dashed = no change)'); plt.show()

## From counts to seats / standing / crowded
Ticket counts are far below true ridership, so we multiply by a **coverage factor** (riders per ticketed passenger). It is an *assumption*: we set the 90th percentile of well-observed weekday-peak stretch loads equal to the standing limit, so the busiest ~10% of peak route-stretches read as 'crowded'. The **ranking** of routes does not depend on it; only the labels do (see the sensitivity table in notebook 4).

In [ ]:
order = list(config.BANDS)
sns.countplot(data=cells, x='band', hue='level', order=order, hue_order=['seats', 'standing', 'crowded'], palette=['#14806A', '#B77900', '#C0332B']); plt.title('Crowd levels by time band'); plt.show()
cells.confidence.value_counts(normalize=True).round(3)

In [ ]:
# Heatmap for the busiest well-observed route: time band (rows) x stretch (columns), weekday
best = cells[(cells.confidence == 'high') & (cells.daytype == 'weekday')].groupby('route_n').load_adj.max().idxmax(); print('busiest well-observed route:', best)
h = cells[(cells.route_n == best) & (cells.daytype == 'weekday')].pivot_table(index='band', columns='pos', values='load_adj')
h = h.reindex([b for b in order if b in h.index]); plt.figure(figsize=(12, 3)); sns.heatmap(h, cmap='YlOrRd'); plt.title(f'Estimated load, route {best}'); plt.show()

In [ ]:
cells.to_parquet(P / 'crowd_segments.parquet'); lv.to_parquet(P / 'crowd_route_level.parquet'); attrs.to_parquet(P / 'route_attrs.parquet')
import json; json.dump(cal, open(P / 'calibration.json', 'w'), indent=2)
model.readable_table(cells, pat, attrs).to_csv(config.OUT / 'crowd_table.csv', index=False); print('saved')